In [ ]:
import os
import requests
import tiktoken
import numpy as np
import random
import torch
import math

In [123]:
input_file_path = './data/tinyshakespeare/input.txt'

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

enc = tiktoken.get_encoding('gpt2')
train_ids = torch.tensor(enc.encode_ordinary(train_data), dtype=torch.long)
val_ids = torch.tensor(enc.encode_ordinary(val_data), dtype=torch.long)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens: {len(val_ids):,}")

train tokens: 301,966
val tokens: 36,059


In [124]:
train_ids[2]

tensor(25)

In [125]:
len(enc._mergeable_ranks) + len(enc._special_tokens), enc.n_vocab

(50257, 50257)

In [126]:
T = block_size = 32
vocab_size = enc.n_vocab
L = n_layer = 3
h = n_head = 4
d_m = n_embd = 96
d_k = int(d_m / h)
d_v = int(d_m / h)
d_ff = 384  # 4 * d_m
B = batch_size = 8
# dropout = 0.1

In [127]:
E = (torch.randn((vocab_size, d_m)) * 0.02).requires_grad_()
P = (torch.randn((T, d_m)) * 0.02).requires_grad_()
W_Q = (torch.randn((d_m, d_k * h)) / math.sqrt(d_m)).requires_grad_()
W_K = (torch.randn((d_m, d_k * h)) / math.sqrt(d_m)).requires_grad_()
W_V = (torch.randn((d_m, d_v * h)) / math.sqrt(d_m)).requires_grad_()
M = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
W_attn_out = (torch.randn((h * d_v, d_m)) / math.sqrt(d_m)).requires_grad_()
W_1 = (torch.randn((d_m, d_ff)) / math.sqrt(d_m)).requires_grad_()
b_1 = torch.zeros((1, d_ff), requires_grad=True)
W_2 = (torch.randn((d_ff, d_m)) / math.sqrt(d_ff)).requires_grad_()
b_2 = torch.zeros((1, d_m), requires_grad=True)
W_O = (torch.randn((d_m, vocab_size)) / math.sqrt(d_m)).requires_grad_()
b_O = torch.zeros((1, vocab_size), requires_grad=True)


In [128]:
def dprint(*args, **kwargs):
    debug = False
    if debug:
        print(*args, **kwargs)

In [145]:
for e in range(100):
    Xs = []
    Ys = []
    for _ in range(batch_size):
        i = random.randrange(len(train_ids) - T)
        x = train_ids[i : i + T]
        y = train_ids[i + 1 : i + T + 1]
        Xs.append(x)
        Ys.append(y)
    X = torch.stack(Xs)
    dprint(f"X = {X.shape}")
    Y = torch.stack(Ys)  # (B, T)
    dprint(f"Y = {Y.shape}")
    # forward
    tok_emb = E[X]
    pos_emb = P
    X1 = tok_emb + pos_emb
    dprint(f"Emb = {X1.shape}")
    # positional encoding
    Q = X1 @ W_Q
    K = X1 @ W_K
    V = X1 @ W_V
    Q = Q.reshape((B, T, h, d_k)).permute(0, 2, 1, 3)
    K = K.reshape((B, T, h, d_k)).permute(0, 2, 1, 3)
    V = V.reshape((B, T, h, d_v)).permute(0, 2, 1, 3)
    dprint(f"Q = {Q.shape}")
    dprint(f"K = {K.shape}")
    dprint(f"V = {V.shape}")
    A = torch.softmax((Q @ K.transpose(-2, -1)) / math.sqrt(d_k) + M, dim=-1) @ V
    A = A.permute(0, 2, 1, 3)  # (B, T, h, d_v)
    A = A.reshape(B, T, h * d_v)  # (B, T, h * d_v)
    A = A @ W_attn_out
    dprint(f"A = {A.shape}")
    X2 = A + X1
    X3 = (X2 - X2.mean(dim=-1, keepdim=True)) / torch.sqrt(X2.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    X4 = torch.relu(X3 @ W_1 + b_1)
    X5 = X4 @ W_2 + b_2
    X6 = X5 + X3
    X7 = (X6 - X6.mean(dim=-1, keepdim=True)) / torch.sqrt(X6.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    logits = X7 @ W_O + b_O
    probs = torch.softmax(logits, dim=-1)
    loss = -torch.log(probs.gather(dim=-1, index=Y.unsqueeze(-1)).squeeze(-1)).mean()
    print(f"loss: {loss.item()}")

    # backward
    for param in (E, P, W_Q, W_K, W_V, W_attn_out, W_1, b_1, W_2, b_2, W_O, b_O):
        param.grad = None
    loss.backward()

    # update
    lr = 0.2
    for param in (E, P, W_Q, W_K, W_V, W_attn_out, W_1, b_1, W_2, b_2, W_O, b_O):
        param.data -= lr * param.grad

loss: 5.160637378692627
loss: 4.823380947113037
loss: 5.044859886169434
loss: 5.516528606414795
loss: 5.5251078605651855
loss: 4.893699645996094
loss: 5.338498115539551
loss: 5.484301567077637
loss: 4.806857109069824
loss: 5.539523124694824
loss: 5.077574729919434
loss: 5.8232316970825195
loss: 5.135303497314453
loss: 5.348066329956055
loss: 4.8114542961120605
loss: 5.74314022064209
loss: 5.559660911560059
loss: 5.685664653778076
loss: 5.158313751220703
loss: 5.3147172927856445
loss: 5.488051414489746
loss: 5.059503555297852
loss: 5.41209602355957
loss: 5.0549468994140625
loss: 5.049156665802002
loss: 5.709253311157227
loss: 4.8803205490112305
loss: 5.322332859039307
loss: 5.009866237640381
loss: 5.340430736541748
loss: 5.057801246643066
loss: 5.365208625793457
loss: 5.052121162414551
loss: 5.136859893798828
loss: 5.410780906677246
loss: 5.222988128662109
loss: 5.257833957672119
loss: 4.786540985107422
loss: 5.230813980102539
loss: 5.491779327392578
loss: 5.592251777648926
loss: 5.6399

In [68]:
K.shape, K.T.shape, K.transpose(1, 2).shape

(torch.Size([8, 32, 96]), torch.Size([96, 32, 8]), torch.Size([8, 96, 32]))

In [148]:
probs.shape

torch.Size([8, 32, 50257])

In [149]:
tokens = [enc.decode([int(i)]) for i in probs[0].argmax(dim=1)]

In [150]:
tokens

['\n',
 'And',
 ' I',
 ' are',
 ' be',
 ' the',
 ' good',
 ',',
 ' the',
 ' king',
 ',',
 '\n',
 '\n',
 '\n',
 'AN',
 'US',
 ':',
 '\n',
 '\n',
 ' are',
 ' a',
 ',',
 ',',
 ',',
 '\n',
 ',',
 ',',
 '\n',
 '\n',
 ' you',
 ',',
 ' grace']

In [151]:
X

tensor([[   11,   198,  2215,   484,   815,  4691,   511, 18901,   287,   262,
          7421,    30,   198,   198,  2257,  1565, 25173,    25,   198,  2990,
           423,   407,   587, 22419,    11, 18680, 18901,    25,   198,  5492,
           340,   534],
        [  198, 34360,    26,   645,    11,   262,  6131, 34360,   714,   407,
          1445,   345,    25,   339, 33041,   198, 28116,   282, 31796,  5443,
           621,   345,  1183,  1560,  1637,    26,   339,   198, 46973,   606,
           355,   339],
        [ 1310,   778,   803,  1517,    25,   438,    46,    11,   612,   198,
           271,   257, 15581,   805,   287,  3240,    11,   530,  6342,    11,
           326,   561,   277,   391,   198, 10724,  9845, 15500,    26,   475,
           673,    11],
        [ 1640,   428,  3708,  2680,  1842,   318,   588,   257,  1049,  3288,
            11,   198,  5562,  4539,   300,   692,   278,   510,   290,   866,
           284,  7808,   465, 26605, 26664,   287,   257,  

In [153]:
[enc.decode([int(i)]) for i in X[0]]

[',',
 '\n',
 'When',
 ' they',
 ' should',
 ' serve',
 ' their',
 ' sovereign',
 ' in',
 ' the',
 ' west',
 '?',
 '\n',
 '\n',
 'ST',
 'AN',
 'LEY',
 ':',
 '\n',
 'They',
 ' have',
 ' not',
 ' been',
 ' commanded',
 ',',
 ' mighty',
 ' sovereign',
 ':',
 '\n',
 'Please',
 ' it',
 ' your']

In [97]:
len(train_ids)

301966

In [101]:
t = random.randrange(len(train_ids - T))

In [102]:
t

80849

In [106]:
vocab_size

50257